In [ ]:
%pip install --upgrade openai pydantic mermaid-py


# Multi-Source Customer Support Agent

Build an AI-powered support agent that queries both a SQL database and a knowledge base of Markdown documents to answer customer questions using OpenAI's Responses API.

## What You Will Build

By the end of this notebook you will have a working customer support agent that can pull answers from **two fundamentally different data sources** in a single conversation:

1. **Structured data (SQLite)** - Customer records, order history, and product inventory stored in relational tables. The agent queries these through parameterized functions (never raw SQL).
2. **Unstructured data (Knowledge Base)** - Markdown documents covering return policies, shipping info, FAQs, and pricing plans. The agent searches these by keyword.

The agent has **3 tools** at its disposal:

| Tool | Data Source | Purpose |
|------|------------|--------|
| `search_orders` | SQLite `orders` + `customers` | Look up orders by email, ID, or status |
| `search_products` | SQLite `products` | Find products by category, keyword, or stock |
| `search_knowledge_base` | Markdown docs | Search policies, FAQs, and documentation |

The key insight is that the **model never generates SQL directly**. Each tool accepts structured parameters (email, order ID, category, etc.) and the Python function constructs the appropriate query internally. This is safer and more predictable than text-to-SQL approaches.

In [1]:
import mermaid as md

md.Mermaid("""
flowchart TD
    A[User Question] --> B[GPT-5-mini Agent]
    B --> C[search_orders]
    B --> D[search_products]
    B --> E[search_knowledge_base]
    C --> F[(SQLite orders/customers)]
    D --> G[(SQLite products)]
    E --> H[(Markdown Docs)]
""")

## Setup

Let's start by importing libraries and connecting to the OpenAI API.

In [2]:
import os
from getpass import getpass
import json
import sqlite3
import pathlib
from typing import Literal

from openai import OpenAI
from pydantic import BaseModel, ConfigDict, Field

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-5-mini"

## Load Data from Resources

Instead of hardcoding data directly in the notebook, we load everything from the `resources/` folder. This keeps the notebook clean and makes it easy to swap in your own data later. You will load three things:

- **Sample data** (JSON): customers, products, and orders
- **Knowledge base** (Markdown files): return policy, shipping info, FAQ, pricing
- **System instructions** (text file): the prompt that tells the agent how to behave

In [3]:
RESOURCES = pathlib.Path("resources")

# Load sample data (customers, products, orders)
data = json.loads((RESOURCES / "data" / "customer_support_sample.json").read_text())

print(f"Loaded {len(data['customers'])} customers, "
      f"{len(data['products'])} products, "
      f"{len(data['orders'])} orders")

# Load knowledge base markdown files
knowledge_base = {}
for md_file in sorted((RESOURCES / "data" / "knowledge_base").glob("*.md")):
    knowledge_base[md_file.name] = md_file.read_text()

print(f"\nKnowledge base contains {len(knowledge_base)} documents:")
for filename, content in knowledge_base.items():
    line_count = len(content.strip().splitlines())
    print(f"  - {filename} ({line_count} lines)")

# Load system instructions
instructions = (RESOURCES / "prompts" / "customer_support_agent_instructions.txt").read_text()
print(f"\nSystem instructions ({len(instructions)} chars):")
print(instructions[:200])

Loaded 8 customers, 10 products, 10 orders

Knowledge base contains 4 documents:
  - faq.md (16 lines)
  - pricing_plans.md (22 lines)
  - return_policy.md (19 lines)
  - shipping_info.md (17 lines)

System instructions (314 chars):
You are a helpful customer support agent for an online store.
Use the available tools to look up information before answering.
Always cite which source you used (order database, product catalog, or kn


## Create the SQLite Database

Now let's create an in-memory SQLite database and populate it with the data we just loaded. Notice how the table schemas enforce constraints (valid plan types, valid order statuses) to keep the data clean.

In [4]:
# Create an in-memory SQLite database
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row  # Return rows as dictionaries
cursor = conn.cursor()

# Create tables
cursor.executescript("""
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    plan_type TEXT NOT NULL CHECK(plan_type IN ('free', 'pro', 'enterprise')),
    created_at TEXT NOT NULL
);

CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL,
    in_stock INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    product_name TEXT NOT NULL,
    quantity INTEGER NOT NULL,
    total_price REAL NOT NULL,
    status TEXT NOT NULL CHECK(status IN ('pending', 'shipped', 'delivered', 'refunded')),
    created_at TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(id)
);
""")

# Populate from loaded JSON data
for c in data["customers"]:
    cursor.execute(
        "INSERT INTO customers VALUES (?, ?, ?, ?, ?)",
        (c["id"], c["name"], c["email"], c["plan_type"], c["created_at"]),
    )

for p in data["products"]:
    cursor.execute(
        "INSERT INTO products VALUES (?, ?, ?, ?, ?)",
        (p["id"], p["name"], p["category"], p["price"], p["in_stock"]),
    )

for o in data["orders"]:
    cursor.execute(
        "INSERT INTO orders VALUES (?, ?, ?, ?, ?, ?, ?)",
        (o["id"], o["customer_id"], o["product_name"],
         o["quantity"], o["total_price"], o["status"], o["created_at"]),
    )

conn.commit()

# Verify the data
for table in ["customers", "products", "orders"]:
    count = cursor.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

customers: 8 rows
products: 10 rows
orders: 10 rows


## Define Pydantic Tool Argument Models

Here is where things get interesting. Instead of hand-typing JSON schemas for each tool (which is tedious and error-prone), we define **Pydantic models** that describe the arguments for each tool. Pydantic can then auto-generate the exact JSON schema that OpenAI expects.

This approach gives you:
- **Validation for free**: Pydantic checks that the model's arguments are valid
- **Self-documenting code**: the Field descriptions double as both code docs and tool descriptions
- **No manual JSON wrangling**: one source of truth for your schema

We start with a `ToolArgs` base class that sets `extra="forbid"` so the generated schema always includes `additionalProperties: false` (required for strict mode).

In [5]:
class ToolArgs(BaseModel):
    model_config = ConfigDict(extra='forbid')

def function_tool(name: str, description: str, args_model: type[BaseModel]) -> dict:
    return {
        "type": "function",
        "name": name,
        "description": description,
        "parameters": args_model.model_json_schema()
    }

class Test(BaseModel):
    test: str
    test_property_two: int

In [6]:
function_tool('test', 'data', Test)

{'type': 'function',
 'name': 'test',
 'description': 'data',
 'parameters': {'properties': {'test': {'title': 'Test', 'type': 'string'},
   'test_property_two': {'title': 'Test Property Two', 'type': 'integer'}},
  'required': ['test', 'test_property_two'],
  'title': 'Test',
  'type': 'object'}}

In [7]:
class SearchOrdersArgs(ToolArgs):
    customer_email: str | None = Field(None, description='customer email address to look up orders for.')
    order_id: int | None = Field(None, description='Specific order ID to look up')
    status: Literal['pending', 'shipped', 'delivered', 'refunded'] = Field(None, description='Filter orders by status')

class SearchProductsArgs(ToolArgs):
    category: str | None = Field(None, description='Product category (e.g. Electronics, Accessories, Furniture)')
    search_term: str | None = Field(None, description='Keyword to search for in product names')
    in_stock_only: bool = Field(False, description='If true, only return products currently in stock')

class SearchKnowledgeBaseArgs(ToolArgs):
    query: str = Field(..., description='Search keywords describing what the customer is asking about')

## Build Tool Functions

Each tool function accepts **structured parameters** and internally constructs the appropriate SQL query or search logic. The model never sees or generates raw SQL; it simply passes parameters like `customer_email` or `category`, and the function handles the rest.

This is a deliberate design choice: parameterized queries are safer, more predictable, and easier to test than text-to-SQL approaches.

In [8]:
def search_orders(**kwargs):

    args = SearchOrdersArgs(**kwargs)

    # Create the SQL query:
    query = """
        SELECT 
            o.id AS order_id,
            c.name as customer_name,
            c.email,
            o.product_name,
            o.quantity,
            o.total_price,
            o.status,
            o.created_at
        FROM orders o
        JOIN customers c ON o.customer_id = c.id
        WHERE 1=1
    """

    params = []

    if args.customer_email:
        query += " AND c.email = ?"
        params.append(args.customer_email)
    if args.order_id is not None:
        query += " AND o.id = ?"
        params.append(args.order_id)
    if args.status:
        query += " AND o.status = ?"
        params.append(args.status)

    query += ' ORDER BY o.created_at DESC'

    # Execute the query:
    rows = cursor.execute(query, params).fetchall()
    results = [dict(row)for row in rows]
    if not results:
        return json.dumps({"message": "No orders found matching the criteria."})
    return json.dumps(results, indent=2)

In [9]:
def search_products(**kwargs):

    args = SearchProductsArgs(**kwargs)

    query = """
        SELECT 
            id,
            name,
            category,
            price,
            in_stock
        FROM products
        WHERE 1=1
    """

    params = []

    if args.category:
        query += " AND LOWER(category) = LOWER(?)"
        params.append(args.category)
        
    if args.search_term:
        query += " AND LOWER(name) LIKE LOWER(?)"
        params.append(f"%{args.search_term}%")

    if args.in_stock_only:
        query += " AND in_stock = 1"
        
    query += " ORDER BY price ASC"

    rows = cursor.execute(query, params).fetchall()
    results = [dict(row) for row in rows]
    if not results:
        return json.dumps({"message": "No products found matching the criteria."})
    return json.dumps(results, indent=2)

In [10]:
knowledge_base

{'faq.md': '# Frequently Asked Questions\n\n## How do I reset my password?\nGo to the login page and click "Forgot Password". Enter your email address and we will send you a reset link. The link expires after 24 hours.\n\n## How do I update my billing information?\nLog in to your account, go to Settings > Billing, and update your payment method. Changes take effect on your next billing cycle.\n\n## Can I change my plan?\nYes, you can upgrade or downgrade your plan at any time from Settings > Subscription. Upgrades take effect immediately; downgrades take effect at the end of your current billing period.\n\n## How do I contact support?\nYou can reach us via this chat agent, by emailing support@example.com, or by calling 1-800-555-0199 during business hours (Mon-Fri, 9 AM - 6 PM EST).\n\n## Do you offer bulk discounts?\nYes, Enterprise plan customers receive volume-based pricing. Contact our sales team at sales@example.com for a custom quote.\n',
 'pricing_plans.md': '# Pricing Plans\n\n

In [11]:
def search_knowledge_base(**kwargs):

    args = SearchKnowledgeBaseArgs(**kwargs)

    # Keywords:
    keywords = args.query.lower().split()
    matches = []

    for filename, content in knowledge_base.items():
        content_lower = content.lower()
        matched_keywords = [kw for kw in keywords if kw in content_lower]

        if matched_keywords:
            matches.append({
                "document": filename,
                "matched_keywords": matched_keywords,
                "relevance": len(matched_keywords) / len(keywords),
                "content": content,
            })

    matches.sort(key=lambda x: x["relevance"], reverse=True)

    if not matches:
        return json.dumps({"message": "No relevant documents found."})
    
    return json.dumps(matches, indent=2)

In [12]:
# search_orders(order_id=1003)
# search_products(category="Electronics")
search_knowledge_base(query="return policy")

'[\n  {\n    "document": "return_policy.md",\n    "matched_keywords": [\n      "return",\n      "policy"\n    ],\n    "relevance": 1.0,\n    "content": "# Return Policy\\n\\nWe offer a 30-day return policy on all products.\\n\\n## Eligibility\\n- Items must be unused and in original packaging.\\n- Electronics must include all original accessories.\\n- Furniture items must be unassembled or in original condition.\\n\\n## Process\\n1. Contact support with your order ID.\\n2. We will email you a prepaid return shipping label.\\n3. Ship the item within 7 days of receiving the label.\\n4. Refund is processed within 5 business days after we receive the item.\\n\\n## Exceptions\\n- Sale items are final sale and cannot be returned.\\n- Customized products are non-returnable.\\n- Items damaged by the customer are not eligible for return.\\n"\n  }\n]'

## Generate Tool Schemas

Now for the payoff. With a single call to `function_tool()` per tool, Pydantic auto-generates the full JSON schema that OpenAI needs. No more hand-typing `"type": "object"`, `"properties": {...}`, and hoping you got it right.

Run the cell below and take a look at the output. Can you spot where your Pydantic `Field(description=...)` values ended up?

In [13]:
tools = [
    function_tool("search_orders", "Search for customer orders by email, order ID, or status.", SearchOrdersArgs),
    function_tool("search_products", "Search the product catalog by category, name, or stock.", SearchProductsArgs),
    function_tool("search_knowledge_base", "Search support docs (return policy, shipping, FAQ, pricing).", SearchKnowledgeBaseArgs),
]

print(f"Registered {len(tools)} tools: {[t['name'] for t in tools]}")
print(tools)

Registered 3 tools: ['search_orders', 'search_products', 'search_knowledge_base']
[{'type': 'function', 'name': 'search_orders', 'description': 'Search for customer orders by email, order ID, or status.', 'parameters': {'additionalProperties': False, 'properties': {'customer_email': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'customer email address to look up orders for.', 'title': 'Customer Email'}, 'order_id': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'description': 'Specific order ID to look up', 'title': 'Order Id'}, 'status': {'default': None, 'description': 'Filter orders by status', 'enum': ['pending', 'shipped', 'delivered', 'refunded'], 'title': 'Status', 'type': 'string'}}, 'title': 'SearchOrdersArgs', 'type': 'object'}}, {'type': 'function', 'name': 'search_products', 'description': 'Search the product catalog by category, name, or stock.', 'parameters': {'additionalProperties': False, 'properties': {'category'

## Tool Dispatch

When the model calls a tool, we need to route that call to the right Python function. The dispatcher below handles argument parsing and filters out `None` values so the function defaults kick in properly.

Notice how this is just a simple dictionary lookup. If you add a new tool later, you only need to add one entry here.

In [14]:
TOOLS_FUNCTIONS = {
    "search_orders": search_orders,
    "search_products": search_products,
    "search_knowledge_base": search_knowledge_base,
}

def dispatch_tool_call(name: str, args: dict):
    func = TOOLS_FUNCTIONS.get(name)
    if func is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        filtered_args = {k: v for k, v in args.items() if v is not None}
        return func(**filtered_args)
    except Exception as e:
        return json.dumps({"error": str(e)})

## The Agent Loop

This is the heart of our agent. The loop follows a simple pattern:

1. Send the user's question to the model along with the available tools.
2. If the model returns a **text response**, we are done; that is the final answer.
3. If the model returns one or more **tool calls**, execute each one and feed the results back.
4. Repeat until the model produces a final text answer or we hit the turn limit.

This loop allows the model to chain multiple tool calls together. For example, it might first search for a customer's orders, then look up the return policy, and finally synthesize both pieces of information into a single answer.

In [17]:
def run_agent(user_query, max_turns=10):
    # Messages Python list:
    messages = [{"role": "user", "content": user_query}]
    number_of_turns = 0
    
    while number_of_turns <= max_turns:
        response = client.responses.create(
            model=MODEL,
            instructions=instructions,
            input=messages,
            tools=tools
        )

        # Increment the number of turns:
        number_of_turns += 1
        print(f" Number of turns: {number_of_turns}")

        # Separate out the tool calls:
        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            print("We finished here as no tool calls left to make")
            return response.output_text

        # Append entire response.output (reasoning + function_calls) - required for reasoning models
        messages.extend(response.output)

        for tool_call in tool_calls:
            args = json.loads(tool_call.arguments)
            print(args)
            result = dispatch_tool_call(tool_call.name, args)
            print(result)
            messages.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": result
            })
    return response.output_text

In [18]:
query = "What is the status of order 1003?"
run_agent(query)

 Number of turns: 1
{'customer_email': None, 'order_id': 1003, 'status': 'pending'}
[
  {
    "order_id": 1003,
    "customer_name": "Bob Smith",
    "email": "bob@example.com",
    "product_name": "USB-C Hub",
    "quantity": 2,
    "total_price": 99.98,
    "status": "pending",
    "created_at": "2024-09-01"
  }
]
 Number of turns: 2
We finished here as no tool calls left to make


'I checked the order database. Order 1003 is currently marked as: pending.\n\nOrder details (from the order database)\n- Order ID: 1003\n- Customer: Bob Smith (bob@example.com)\n- Item: USB-C Hub (qty 2)\n- Total: $99.98\n- Status: pending\n- Created: 2024-09-01\n\nWould you like me to (pick one)?\n- Check an estimated shipping date or expected fulfillment time\n- Cancel or modify the order (I can start that request)\n- Escalate to fulfillment for faster handling\n\nTell me which and I’ll proceed.'

## Try It Out

Time to see the agent in action! We will run a series of queries that exercise each tool individually and then test multi-tool scenarios where the agent needs to combine information from different sources.

### Query 1: Order Lookup by ID

Let's start simple. The agent should use `search_orders` to find this order.

In [ ]:
query = "What's the status of order #1003?"
print(f"{'=' * 60}")
print(f"QUERY: {query}")
print(f"{'=' * 60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")


### Query 2: Product Search by Category

Now let's try a product catalog search. Watch how the agent uses `search_products` with the `category` parameter.

In [ ]:
query = "What products do you have in the Electronics category?"
print(f"{'=' * 60}")
print(f"QUERY: {query}")
print(f"{'=' * 60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")


### Query 3: Knowledge Base Search

This question should trigger `search_knowledge_base` since it is about a policy, not about a specific order or product.

In [ ]:
query = "What is your return policy?"
print(f"{'=' * 60}")
print(f"QUERY: {query}")
print(f"{'=' * 60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")


### Query 4: Multi-Tool Query

Here is where it gets exciting. This question requires the agent to use **both** `search_orders` (to find Alice's orders) **and** `search_knowledge_base` (to check the return policy). Watch the verbose output to see how the agent chains multiple tool calls together.

In [ ]:
query = (
    "I'm alice@example.com and I want to know about my orders "
    "and whether I can return any of them."
)
print(f"{'=' * 60}")
print(f"QUERY: {query}")
print(f"{'=' * 60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")


### Query 5: Constrained Product Search

What happens when the user asks about something that does not match any products? Try changing the search term below to see how the agent handles empty results gracefully.

In [ ]:
query = "Do you have any laptops in stock under $1000?"
print(f"{'=' * 60}")
print(f"QUERY: {query}")
print(f"{'=' * 60}")
answer = run_agent(query)
print(f"\nANSWER:\n{answer}\n")


## Your Turn: Add a Warranty Check Tool

Your agent already has three tools. Now you will add a fourth: `check_warranty`. This tests your ability to define a new Pydantic args model, write the tool function, and wire it into the existing agent.

## Exercise 1: Add a New Tool to the Customer Support Agent

Your turn! The customer support agent from the previous notebook already has three working tools:

| Tool | Purpose |
|------|---------|
| `search_orders` | Look up orders by email, order ID, or status |
| `search_products` | Find products by category, keyword, or stock availability |
| `search_knowledge_base` | Search support docs (return policy, shipping, FAQ, pricing) |

Your job is to **add a fourth tool** called `check_warranty` that tells the customer whether a product is still under warranty.

**Warranty rules:**
- **Electronics**: 1 year warranty
- **Furniture**: 2 year warranty
- **Accessories**: 6 month warranty
- If the product category is not recognized, return "No warranty information available."

The tool should accept two parameters:
- `product_name` (str): the name of the product to check
- `purchase_date` (str): the date the product was purchased, in `YYYY-MM-DD` format

It should look up the product in the database to determine its category, apply the warranty rules, and return a JSON result with the warranty status.

### Your Code: Define `CheckWarrantyArgs` and `check_warranty`

Now you get to build the fourth tool! Follow these steps:

1. **Create the Pydantic model** `CheckWarrantyArgs` that extends `ToolArgs`. It should have two required fields: `product_name` (str) and `purchase_date` (str). Add helpful `description` strings to each field.
2. **Write the `check_warranty` function.** It should:
   - Look up the product in the database by name (case-insensitive) to find its category.
   - Calculate whether the warranty has expired based on the purchase date and today's date.
   - Return a JSON string with the product name, category, purchase date, warranty duration, expiry date, and whether the warranty is still active.
3. If the product is not found in the database, return a JSON error message.

In [ ]:
# YOUR CODE HERE: Define CheckWarrantyArgs
class CheckWarrantyArgs(ToolArgs):
    pass  # Replace with your fields


# YOUR CODE HERE: Implement check_warranty
def check_warranty(product_name, purchase_date):
    """Check warranty status for a product based on its category and purchase date."""
    # Warranty durations by category (in months)
    # Electronics: 12 months, Furniture: 24 months, Accessories: 6 months

    pass  # Your implementation here

### Your Code: Register the New Tool and Update Dispatch

Now wire your new tool into the agent. You need to:

1. Add a `function_tool(...)` entry for `check_warranty` to the `tools` list.
2. Add `check_warranty` to the `TOOL_FUNCTIONS` dispatch dictionary.

In [ ]:
# Build the tools list (the first 3 are done for you)
tools = [
    function_tool(
        "search_orders",
        "Search for customer orders by email, order ID, or status.",
        SearchOrdersArgs,
    ),
    function_tool(
        "search_products",
        "Search the product catalog by category, name, or stock.",
        SearchProductsArgs,
    ),
    function_tool(
        "search_knowledge_base",
        "Search support docs (return policy, shipping, FAQ, pricing).",
        SearchKnowledgeBaseArgs,
    ),
    # YOUR CODE HERE: Add the check_warranty tool
]

# Build the dispatch dictionary (the first 3 are done for you)
TOOL_FUNCTIONS = {
    "search_orders": search_orders,
    "search_products": search_products,
    "search_knowledge_base": search_knowledge_base,
    # YOUR CODE HERE: Add check_warranty
}


def dispatch_tool_call(name, args):
    """Route a tool call to the appropriate function."""
    func = TOOL_FUNCTIONS.get(name)
    if func is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        filtered_args = {k: v for k, v in args.items() if v is not None}
        return func(**filtered_args)
    except Exception as e:
        return json.dumps({"error": str(e)})


print(f"Registered {len(tools)} tools: {[t['name'] for t in tools]}")

### Test Your Work

Run the test cell below. If your `check_warranty` tool is implemented correctly, the agent should look up the product, determine its category, and tell the customer whether the warranty is still active.

In [ ]:
answer = run_agent("Is my Ergonomic Chair from 2024-06-20 still under warranty?")
print(f"\nFINAL ANSWER:\n{answer}")

In [ ]:
# Bonus: try a product from a different category
answer = run_agent("I bought a Wireless Mouse on 2024-01-15. Is it still covered?")
print(f"\nFINAL ANSWER:\n{answer}")

## Summary

Great work! In this notebook you built a multi-source customer support agent that combines structured and unstructured data retrieval. Here are the key takeaways:

- **Multi-source retrieval**: Real support agents need access to different kinds of data. Your agent seamlessly queries a SQL database for transactional data and searches Markdown documents for policy information, choosing the right tool for each question.

- **Pydantic-powered schemas**: Instead of hand-typing JSON schemas, you used Pydantic models with `model_json_schema()` to auto-generate tool definitions. This keeps your code DRY, self-documenting, and less error-prone.

- **Structured parameters over text-to-SQL**: Instead of letting the model generate raw SQL (which is fragile and risky), you defined tools with typed, constrained parameters. The model picks values like `category="Electronics"` or `order_id=1003`, and the Python functions build safe queries internally.

- **The agent loop pattern**: The core loop (call the model, check for tool calls, execute tools, feed results back) is the fundamental building block of agentic systems. It allows the model to chain multiple lookups before composing a final answer.

- **Combining data sources in one response**: The most powerful queries are those that require information from multiple sources (e.g., looking up a customer's orders AND checking the return policy). The agent loop naturally handles this by allowing the model to make sequential or parallel tool calls across different data sources.